In [ ]:
import requests
import tempfile
import pypdf
import re
from typing import List, Tuple, Set

def is_artifact(line: str) -> bool:
    """Check if a line is likely a formatting artifact."""
    punctuation = ".,:;?!-()[]{}'/\"$%"
    tokens = [t for t in line.split() if t]
    is_single_token_no_punctuation = len(tokens) == 1 and not any(p in tokens[0] for p in punctuation)
    if is_single_token_no_punctuation:
        return True

    artifacts = [
        r"Fmt \d+",
        r"Frm \d+",
        r"Sfmt \d+",
        r"VerDate.*",
        r"HR \d+ .*",
        r'^\s*Page\s+\d+\s*$',
        r'^\s*-\d+-\s*$',
        r'^\s*\[\d+\]\s*$',
        r'^\s*\(continued\)\s*$',
        r'^\s*\d+\s*$',
        r'^\s*[A-Z]\s*$'
    ]
    return any(re.search(pattern, line, re.IGNORECASE) for pattern in artifacts)

def is_all_caps(text: str) -> bool:
    """Check if text is in all capitals, ignoring non-letters."""
    letters = ''.join(c for c in text if c.isalpha())
    return letters and letters.isupper()

def is_title_case(text: str) -> bool:
    """Check if text follows title case rules."""
    exceptions: Set[str] = {"of", "the", "and"}
    words = text.split()
    if not words:
        return False

    # First word should always be capitalized
    if not words[0][0].isupper():
        return False

    # Check subsequent words
    for i, word in enumerate(words[1:], 1):
        if not word:
            continue
        # Skip checking exceptions unless it's the last word
        if word.lower() in exceptions:
            continue
        if not word[0].isupper() and not any(p in word for p in "'\"([{"):
            return False
    return True

def is_connecting_phrase(line: str) -> bool:
    """Check if line is a connecting phrase like 'of the'."""
    connecting_phrases = {
        "of the", "in the", "to the", "and the", "for the",
        "with the", "by the", "on the", "at the", "and"
    }
    return line.strip().lower() in connecting_phrases

def get_case_state(text: str) -> str:
    """Determine the case state of text."""
    if is_all_caps(text):
        return "caps"
    elif is_title_case(text):
        return "title"
    return "normal"

def identify_section_headers(line: str) -> Tuple[bool, int]:
    """
    Identify if a line is a section header and its level.
    Returns (is_header, header_level).
    """
    patterns = [
        (r'^[A-Z][A-Z\s]+$', 1),  # ALL CAPS -> h1
        (r'^\d+\.\s+[A-Z]', 2),   # Numbered sections -> h2
        (r'^[IVXLCDM]+\.\s+[A-Z]', 2),  # Roman numeral sections -> h2
        (r'^\(\d+\)\s+[A-Z]', 3),  # (1) Style sections -> h3
        (r'^[a-z]\)\s+[A-Z]', 3)   # a) Style sections -> h3
    ]

    for pattern, level in patterns:
        if re.match(pattern, line.strip()):
            return True, level

    if is_all_caps(line):
        return True, 1

    if is_title_case(line) and len(line.split()) > 1:
        return True, 2

    return False, 0

def clean_and_format_text(text: str) -> str:
    """
    Clean extracted text and format as HTML.
    Handles multi-line titles, hyphenated words, and different text cases.

    Args:
        text: Input text to clean and format

    Returns:
        Cleaned and formatted text in HTML format
    """
    lines = text.split("\n")
    html_parts: List[str] = []
    current_paragraph: List[str] = []
    last_case_state = None
    hyphenated_word = ""

    def get_line_at_index(index: int) -> str:
        """Safely get line at index, return empty string if out of bounds."""
        if 0 <= index < len(lines):
            return lines[index].strip()
        return ""

    def check_for_title(current_idx: int) -> Tuple[bool, bool]:
        """
        Check if current line is part of a multi-line title.
        Returns (is_title_component, is_terminating_component).
        """
        current_line = get_line_at_index(current_idx)
        prev_line = get_line_at_index(current_idx - 1)
        next_line = get_line_at_index(current_idx + 1)

        print(prev_line)
        print(current_line)
        print(next_line)

        print(get_case_state(current_line))
        if get_case_state(current_line) in ["caps", "title"]:
            if is_connecting_phrase(prev_line):
                return True, True
            if is_connecting_phrase(next_line):
                return True, False

        if is_connecting_phrase(current_line):
            prev_case = get_case_state(prev_line)
            next_case = get_case_state(next_line)
            return (prev_case in ["caps", "title"] and
                   next_case in ["caps", "title"]), False

        return False, False

    def flush_paragraph() -> List[str]:
        """Flush current paragraph to HTML parts."""
        if current_paragraph:
            # Join with space, handle special cases
            text = " ".join(current_paragraph)
            # Clean up any double spaces
            text = re.sub(r'\s+', ' ', text)
            html_parts.append(f"<p>{text}</p>")
            return []
        return current_paragraph

    def escape_html(text: str) -> str:
        """Escape HTML special characters."""
        return (text.replace("&", "&amp;")
                   .replace("<", "&lt;")
                   .replace(">", "&gt;")
                   .replace('"', "&quot;")
                   .replace("'", "&#39;"))

    # Start HTML document
    html_parts.append('<!DOCTYPE html>')
    html_parts.append('<html>')
    html_parts.append('<head>')
    html_parts.append('<meta charset="UTF-8">')
    html_parts.append('<style>')
    html_parts.append("""
        body {
            font-family: Arial, sans-serif;
            line-height: 1.6;
            max-width: 800px;
            margin: 0 auto;
            padding: 20px;
        }
        h1, h2, h3 { color: #333; }
        p { margin-bottom: 1em; }
    """)
    html_parts.append('</style>')
    html_parts.append('</head>')
    html_parts.append('<body>')

    for i, line in enumerate(lines):
        line = line.strip()
        if not line or is_artifact(line):
            continue

        print('---')
        print(line)

        # Remove line numbers at the start
        line = re.sub(r"^\d+\s+", "", line)

        # Escape HTML special characters
        line = escape_html(line)

        # Handle hyphenated words
        if hyphenated_word:
            if line:
                line = hyphenated_word + line
            hyphenated_word = ""
        if line.endswith('-'):
            hyphenated_word = line[:-1]
            continue

        # Check if this line is part of a multi-line title
        print(1)
        is_title_component, is_terminating_component = check_for_title(i)
        if is_title_component:
            print(is_title_component)
            print(is_terminating_component)
            current_paragraph.append(line)
            if is_terminating_component:
                current_paragraph = flush_paragraph()
            continue
        print(2)

        # Check if it's a header
        is_header, header_level = identify_section_headers(line)
        if is_header:
            current_paragraph = flush_paragraph()
            html_parts.append(f"<h{header_level}>{line}</h{header_level}>")
            continue

        # Handle case state changes
        current_case_state = get_case_state(line)
        if last_case_state is not None and current_case_state != last_case_state:
            current_paragraph = flush_paragraph()
        last_case_state = current_case_state

        # If the line begins with certain characters, start a new paragraph
        if line.startswith(("\u2018\u2018", "(", "''(", "'(")):
            current_paragraph = flush_paragraph()

        # Add line to current paragraph
        current_paragraph.append(line)

    # Flush any remaining content
    if current_paragraph:
        current_paragraph = flush_paragraph()

    # Close HTML document
    html_parts.append('</body>')
    html_parts.append('</html>')

    return "\n".join(html_parts)

def download_and_process_pdf(url: str) -> str:
    """
    Download and process a PDF file, returning HTML-formatted text.
    """
    with tempfile.NamedTemporaryFile(delete=False) as pdf_file:
        # Download the PDF
        CHUNK_SIZE = 32768
        with requests.Session() as session:
            response = session.get(url, stream=True, timeout=30)
            response.raise_for_status()
            for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                if chunk:
                    pdf_file.write(chunk)
        pdf_file.flush()
        temp_filename = pdf_file.name

    # Process the PDF
    html_parts = []
    try:
        with open(temp_filename, "rb") as file:
            pdf = pypdf.PdfReader(file)

            batch_size = 5
            for i in range(0, len(pdf.pages), batch_size):
                batch = pdf.pages[i : i + batch_size]
                for page in batch:
                    try:
                        page_text = page.extract_text()
                        cleaned_html = clean_and_format_text(page_text)
                        if cleaned_html:
                            html_parts.append(cleaned_html)
                    except Exception as e:
                        print(f"Error on page {i}: {e}")
                        continue

                import gc
                gc.collect()

        return "\n".join(html_parts)

    finally:
        # Clean up the temporary file
        import os
        try:
            os.unlink(temp_filename)
        except:
            pass

# Example usage
if __name__ == "__main__":
    url = "https://www.congress.gov/118/bills/s2685/BILLS-118s2685enr.pdf"
    html_text = download_and_process_pdf(url)

    # Save to file
    with open("output.html", "w", encoding="utf-8") as f:
        f.write(html_text)